# Lesson 18: How SmolVLA uses conditions

## The idea before the implementation

**Find the routes by which context affects action generation.** A conditional policy must produce different actions when relevant inputs change. In SmolVLA, images, text, and projected state enter the prefix. The noisy action and flow time enter the action suffix. The expert combines these routes through attention.

**Where you work.** You work on model inspection, not on a new training objective. A key/value cache stores prefix attention tensors for reuse within one prediction. Controlled probes change one condition while keeping action noise and time fixed, then measure the velocity change. A changed output shows sensitivity to that route; it does not by itself show correct task understanding.

**Pause and explain.** Explain why changing the noise as well as the instruction would make a prompt-sensitivity result ambiguous.

Keep [the concept guide](../concepts/smolvla.md) open beside this notebook for the full system diagrams and lesson map.

This lesson uses small code cells to show the real SmolVLA path. Run cells in order. This notebook contains complete exercise answers.

**Result:** a measurable account of how the VLM prefix, key/value cache, state, time, and action expert condition each prediction.

**Notebook pair:** [Learner notebook](18-smolvla-conditioning.ipynb).

**Evidence boundary:** CPU checks show tensor mechanics. They do not show a useful robot policy. Cells with an experiment tag load local model or LIBERO assets and can take much longer.

In [ ]:
import os
import sys
from pathlib import Path


def find_workspace() -> Path:
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the tutorial workspace.")


WORKSPACE = find_workspace()
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"

In [ ]:
from collections.abc import Mapping

import torch
from torch.utils.data import default_collate

from vla_lab.teaching.act_model import frame_offset_seconds, open_libero_dataset
from vla_lab.teaching.interfaces import VLAInputs
from vla_lab.teaching.smolvla import SmolVLAAdapter, load_prepared_smolvla

## Model map: prefix attention and action conditioning

The prefix combines image patches, text tokens, and robot state. The VLM processes this prefix once and can store **key/value cache** tensors. In attention, a query compares with keys to make weights; those weights combine values. For one query, attention computes:

$$\operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d}}\right)V$$

Where:
* $Q$ - the query vector(s).
* $K$ - the key vectors, one per prefix token.
* $V$ - the value vectors, one per prefix token.
* $d$ - the number of values per attention head.

If two key scores are $0$ and $\ln 3$, the weights are $0.25$ and $0.75$; values $0$ and $4$ give context $3$. The cache stores prefix keys and values so each denoising step can reuse visual and language context. A cache belongs to one observation. Rebuild it after the camera image, instruction, or state changes.

The action expert is not the ACT decoder from earlier lessons. It is a small language-model-style stack paired with the VLM. It reads cached prefix keys/values and current action tokens. SmolVLA embeds the noisy action and the time `t` together in its suffix. Robot state is embedded with the observation prefix. These are two separate conditioning routes:

<svg xmlns="http://www.w3.org/2000/svg" width="780" height="250" viewBox="0 0 780 250" role="img" aria-label="Two conditioning routes enter the action expert">
<title>Two conditioning routes enter the action expert</title>
<desc>The expert reads both routes. Hold noise and flow time fixed when testing a changed condition.</desc>
<rect width="780" height="250" rx="8" fill="#f8fafc"/>
<text x="12" y="23" font-family="sans-serif" font-size="15" font-weight="bold" fill="#0f172a">Two conditioning routes enter the action expert</text>
<path d="M 207 71 L 245 71 M 239 67 L 245 71 L 239 75" fill="none" stroke="#334155" stroke-width="2"/>
<path d="M 207 166 L 245 166 M 239 162 L 245 166 L 239 170" fill="none" stroke="#334155" stroke-width="2"/>
<path d="M 425 71 L 455 71 L 455 104 L 490 104 M 484 100 L 490 104 L 484 108" fill="none" stroke="#334155" stroke-width="2"/>
<path d="M 425 166 L 455 166 L 455 130 L 490 130 M 484 126 L 490 130 L 484 134" fill="none" stroke="#334155" stroke-width="2"/>
<path d="M 615 117 L 651 117 M 645 113 L 651 117 L 645 121" fill="none" stroke="#334155" stroke-width="2"/>
<rect x="12" y="45" width="195" height="52" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
<text x="109.5" y="66" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Images, text, state</text>
<text x="109.5" y="83" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">prefix embeddings</text>
<rect x="245" y="45" width="180" height="52" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
<text x="335" y="66" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">VLM processing</text>
<text x="335" y="83" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">cached keys and values</text>
<rect x="12" y="140" width="195" height="52" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
<text x="109.5" y="161" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Noisy actions</text>
<text x="109.5" y="178" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">and flow time</text>
<rect x="245" y="140" width="180" height="52" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
<text x="335" y="161" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Action and time</text>
<text x="335" y="178" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">embeddings</text>
<rect x="490" y="91" width="125" height="52" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
<text x="552.5" y="112" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Action expert</text>
<text x="552.5" y="129" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">attention</text>
<rect x="651" y="91" width="115" height="52" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
<text x="708.5" y="112" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Predicted</text>
<text x="708.5" y="129" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">velocity</text>
<text x="12" y="235" font-family="sans-serif" font-size="12" fill="#334155">The expert reads both routes. Hold noise and flow time fixed when testing a changed condition.</text>
</svg>

The first route supplies cached prefix keys and values from images, text, and state. The second supplies expert queries and action features from the noisy action and time embeddings. Both routes enter expert attention. The two branches meet at the expert, as shown in the diagram.


Shapes:
* prefix embeddings - $(B,L_p,C)$;
* cached keys/values - one pair per layer, each with batch $B$, prefix length $L_p$, and head dimensions;
* action embeddings - $(B,T,C)$;
* time - one scalar per example, $(B,)$.

Where $B$ is the batch size, $L_p$ the number of prefix tokens, $T$ the number of action steps, and $C$ the hidden width. Attention masks prevent padded image/text tokens from contributing. The cache is not a compressed image: it is the attention state needed to reuse this prefix. For the two-key example above, the attention context is $0.25\cdot0+0.75\cdot4=3$; changing the state can change the prefix key/value vectors and therefore this context.

A useful architecture probe changes one condition at a time and measures a velocity difference. This does not prove policy quality. It shows which input route reaches the supplied model. Use identical action noise and time for both measurements so only the chosen condition changes.

## Exercise 18.1 — Measure a conditioning route

**Purpose.** Write a small probe that changes robot-state conditioning while keeping image, text, noisy actions, and time fixed. The model output should change when its state input changes.

Implement `state_condition_delta(adapter, inputs, x_t, time)`. `inputs` is the `VLAInputs` dict from Lesson 17 (`images`, `img_masks`, `lang_tokens`, `lang_masks`, `state`); `x_t` is the noisy action chunk $(B,T,32)$ and `time` is $(B,)$. Return a 0-dimensional tensor:

$$\Delta=\frac{1}{N}\sum\left|v_\theta(x_t,t,s)-v_\theta(x_t,t,\mathbf{0})\right|$$

Where:
* $v_\theta(x_t,t,s)$ - the predicted velocity with the supplied state $s$ = `inputs["state"]`.
* $v_\theta(x_t,t,\mathbf{0})$ - the predicted velocity with an all-zero state tensor of the same shape, device, and dtype; everything else is unchanged.
* $N$ - the number of values in one velocity tensor; the sum runs over all of them. Use `adapter.velocity` for both predictions. Do not change the model weights, mask, noise, or time.

A zero result can be valid for a random model if the selected layer is insensitive. A non-zero result proves only that state can affect the vector field. It does not prove that the policy uses state correctly on a robot.

More behaviour checks for this exercise: `check_state_condition_delta` in [lesson18_checks.py](lesson18_checks.py).

In [ ]:
def state_condition_delta(
    adapter: SmolVLAAdapter, inputs: VLAInputs, x_t: torch.Tensor, time: torch.Tensor
) -> torch.Tensor:
    with_state = adapter.velocity(inputs, x_t, time)
    zero_state_inputs = dict(inputs)
    zero_state_inputs["state"] = torch.zeros_like(inputs["state"])
    without_state = adapter.velocity(zero_state_inputs, x_t, time)
    return (with_state - without_state).abs().mean()


# This number measures sensitivity to state. It does not measure task success.

The CPU check uses a deterministic toy velocity function. The tagged model probe loads the published policy and compares state-on with zero-state using the same image, prompt, noise, and time.

In [ ]:
# Pure tensor check for the expected output contract.
class ToyAdapter:
    def velocity(
        self: "ToyAdapter",
        inputs: Mapping[str, torch.Tensor],
        x_t: torch.Tensor,
        time: torch.Tensor,
    ) -> torch.Tensor:
        return x_t + inputs["state"].mean(dim=-1, keepdim=True).view(-1, 1, 1)


toy = ToyAdapter()
inputs = {"state": torch.tensor([[1.0, 1.0], [2.0, 2.0]])}
delta = state_condition_delta(toy, inputs, torch.zeros(2, 3, 1), torch.tensor([0.2, 0.8]))
assert torch.isclose(delta, torch.tensor(1.5))
print("state-condition delta:", float(delta))

# More behaviour checks: lesson18_checks.py, next to this notebook.
from lesson18_checks import check_state_condition_delta

check_state_condition_delta(state_condition_delta)

**Explain (2–3 sentences).** Why must the cached prefix be rebuilt after the state or image changes? Which parts are held fixed in the probe?

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
POLICY, PREPROCESSOR, POSTPROCESSOR = load_prepared_smolvla("published", DATA, device=DEVICE)
ACTION_CHUNK_SIZE = POLICY.config.chunk_size
DATASET = open_libero_dataset(
    CONFIGS,
    DATA,
    "bowl_plate",
    "validation",
    {"action": frame_offset_seconds(list(range(ACTION_CHUNK_SIZE)))},
)
RAW = default_collate([DATASET[0]])
RAW["task"] = ["put the bowl on the plate"]
BATCH = PREPROCESSOR(RAW)
BATCH = {
    key: value.to(DEVICE) if isinstance(value, torch.Tensor) else value
    for key, value in BATCH.items()
}
ADAPTER = SmolVLAAdapter(POLICY)
INPUTS = ADAPTER.prepare_inputs(BATCH)
NOISE = torch.randn(1, ADAPTER.chunk_size, ADAPTER.max_action_dim, device=DEVICE)
TIME = torch.tensor([0.5], device=DEVICE)
DELTA = state_condition_delta(ADAPTER, INPUTS, NOISE, TIME)
PREFIX_PAD, PREFIX_CACHE = ADAPTER.prefix_cache(INPUTS)
assert PREFIX_CACHE is not None
CACHE_LAYERS = PREFIX_CACHE.layers
assert len(CACHE_LAYERS) > 0 and CACHE_LAYERS[0].keys is not None
print("mean absolute velocity change after zeroing state:", float(DELTA))
print(
    "prefix padding shape:",
    tuple(PREFIX_PAD.shape),
    "| cached layers:",
    len(CACHE_LAYERS),
)
print("first key tensor shape:", tuple(CACHE_LAYERS[0].keys.shape))

This compares one condition while keeping the rest fixed. The plot below is a visual toy model, not the SmolVLA policy and not a training cell.